## How to us Mido to process midi keyboard data in python. 

This is pulled from https://mido.readthedocs.io/en/latest/ports.html

In [1]:
#  !mamba install mido
#  !pip install python-rtmidi
import mido, logging, time, libcsound, numpy as np, pprint as pp, adaptive_tuning_util as atu, diamond_music_utils as dmu
np.set_printoptions(legacy='1.21', precision=3, suppress=True)
dmu.start_logger('test.log', log_level='info')

In [2]:
for inputs in mido.get_input_names():
    print(f'{inputs = }')
for backend in mido.backend.module.get_api_names():
    print(f'{backend = }')

inputs = 'Midi Through:Midi Through Port-0 14:0'
inputs = 'LPK25:LPK25 MIDI 1 28:0'
inputs = 'Midi Through:Midi Through Port-0 14:0'
inputs = 'LPK25:LPK25 MIDI 1 28:0'
backend = 'LINUX_ALSA'
backend = 'UNIX_JACK'


In [3]:
port = mido.open_input('LPK25 MIDI 1')
inx = 0
for msg in port:
    print(msg)
    inx += 1
    if inx > 10: break

note_on channel=0 note=57 velocity=100 time=0
note_off channel=0 note=57 velocity=127 time=0
note_on channel=0 note=55 velocity=93 time=0
note_off channel=0 note=55 velocity=127 time=0
note_on channel=0 note=53 velocity=67 time=0
note_off channel=0 note=53 velocity=127 time=0
note_on channel=0 note=53 velocity=62 time=0
note_on channel=0 note=52 velocity=66 time=0
note_off channel=0 note=52 velocity=127 time=0
note_off channel=0 note=53 velocity=127 time=0
note_on channel=0 note=53 velocity=90 time=0


In [4]:
print(f'{port.name = }')

port.name = 'LPK25:LPK25 MIDI 1 28:0'


### Accept midi notes played on the keyboard and load them to a numpy array
Use this as an example for waiting for notes to be played. 
Later we can figure out how to send them to the running csound instance using libcsound. This will print the midi value of the note and its velocity for any keypresses on the midi keyboard. As soon as it passes 10 seconds, the next keypress will terminate the loop.

In [5]:
port = mido.open_input('LPK25 MIDI 1')
one_hot_note = np.zeros([127]) # one hot encoding of the current chord, with velocity as value
ticks_per_beat = 24 # pulses per quarter note
start = time.time()
print(f'{start = }')
for msg in port:
    print(f'{msg.type = }, {msg.note = }, {msg.velocity = }, {round((time.time() - start) * ticks_per_beat,1)}') # you can refer to values as msg.something
    if msg.type == 'note_on':
        one_hot_note[msg.note] = msg.velocity
    elif msg.type == 'note_off':
        one_hot_note[msg.note] = 0
    if np.any(one_hot_note > 0):
        # convert the one hot encoded array to a list of notes that are currently on, with their velocities
        current_chord = np.array([np.where(one_hot_note > 0)[0], one_hot_note[np.where(one_hot_note > 0)[0]]]).T
        print(f'{current_chord = }\n{time.time() - start = }')
    # why doesn't this end the loop after 10 seconds? Because the loop is waiting for MIDI messages, and if there are no messages, it will just wait indefinitely. We need to check the time inside the loop and break if it's been more than 10 seconds.
    if time.time() > start + 10: 
        print(f'no longer waiting for a MIDI message, 10 seconds have passed, so exiting loop.')
        break
port.close()

start = 1773506874.1271331
msg.type = 'note_on', msg.note = 60, msg.velocity = 89, 87.0
current_chord = array([[60., 89.]])
time.time() - start = 3.6581919193267822
msg.type = 'note_on', msg.note = 57, msg.velocity = 88, 87.8
current_chord = array([[57., 88.],
       [60., 89.]])
time.time() - start = 3.658513307571411
msg.type = 'note_on', msg.note = 53, msg.velocity = 78, 87.8
current_chord = array([[53., 78.],
       [57., 88.],
       [60., 89.]])
time.time() - start = 3.6587321758270264
msg.type = 'note_off', msg.note = 57, msg.velocity = 127, 109.0
current_chord = array([[53., 78.],
       [60., 89.]])
time.time() - start = 4.5415403842926025
msg.type = 'note_off', msg.note = 53, msg.velocity = 127, 109.1
current_chord = array([[60., 89.]])
time.time() - start = 4.544976472854614
msg.type = 'note_off', msg.note = 60, msg.velocity = 127, 109.3
msg.type = 'note_on', msg.note = 60, msg.velocity = 99, 127.5
current_chord = array([[60., 99.]])
time.time() - start = 5.311549663543701
m

### Get csound to play some notes through the libcsound interface/
This uses a default simple csd file of sine waves to play a C major chord in 12-TET.

In [6]:
import libcsound
cs = libcsound.Csound()
cs.setOption('-d -odac -m0')
cs.compileOrc(r'''
sr     = 48000
ksmps  = 64
nchnls = 2
0dbfs  = 1

instr 1
  iamp, ipitch, iattack, idec, ipan passign 4
  aenv = linen:a(1, iattack, p3, idec)
  asig = poscil(iamp, mtof(ipitch)) * aenv
  a1, a2 pan2 asig, ipan
  outs a1, a2
endin
''')

thread = cs.performanceThread()
# Calling thread.play starts the csound process, if not already started
thread.play()
thread.scoreEvent(False, 'i', (1, 0,   2, 0.5, 60, 0.05, 0.3, 0.2))
thread.scoreEvent(False, 'i', (1, 0.5, 2, 0.5, 64, 0.05, 0.3, 0.8))
thread.scoreEvent(False, 'i', (1, 1.0, 2, 0.5, 67, 0.05, 0.3, 0.8))

--Csound version 7.0 (double samples) Feb  3 2026
[commit: 4596d8246d1a9df12004e3abfd8d43d2e89284e7]
using libsndfile-1.2.2
sr = 48000.0, kr = 750.000, ksmps = 64
0dBFS level = 1.0, A4 tuning = 440.0
audio buffered in 256 sample-frame blocks
writing 512 sample blks of 64-bit floats to dac
SECTION 1:


In [7]:
thread.stop()
thread.join()
del cs

		   overall amps:  0.78263  1.10280
	   overall samples out of range:        0      634
0 errors in performance
1058 512 sample blks of 64-bit floats written to dac


### Load the ball10.csd file to play using samples
This uses the sample based synthesis to play notes on the keyboard that are played by a flute sample, with notes specified in cents.

In [8]:
import libcsound
import numpy as np

def initiate_csound():
    cs = libcsound.Csound()
    # cs.setOption('-d m0') # -d is for disabling displays, -odac is for outputting to the audio device, -m0 is for no message file. You can also specify a message file with -m followed by the filename.
    cs.setOption('-d -odac -m0')
    cs.compileCsd('ball10.csd')
    thread = cs.performanceThread()
    # Calling thread.play starts the csound process, if not already started
    thread.play()
    return cs, thread

### What ball10.csd needs to see:
```text
    ; +--- the only instrument in the orchestra is 1
    ; |     +--- always 0, no delay from key press from the note-on command
    ; |     |       +--- should be infinite, until you receive a note-off command
    ; |     |       |       +--- set by the velocity of the note-on command
    ; |     |       |       |    +--- tuned cent value derived from the note-on
    ; |     |       |       |    |   +--- derived from the midi value
    ; |     |       |       |    |   |   +--- set by GUI
    ; |     |       |       |    |   |   |   +--- set by GUI
    ; |     |       |       |    |   |   |   |   +--- set by GUI
    ; |     |       |       |    |   |   |   |   |    +--- ??? predicted next cent value?
    ; |     |       |       |    |   |   |   |   |    |    +--- set by GUI - enforced munchkinization
    ; |     |       |       |    |   |   |   |   |    |    |   +--- set by GUI
    ; |     |       |       |    |   |   |   |   |    |    |   |   +--- vibrato set by gui
    ; |     |       |       |    |   |   |   |   |    |    |   |   |   +--- secondary vibrato by gui
    ; |     |       |       |    |   |   |   |   |    |    |   |   |   |   +--- set by gui
    ; 1     2       3       4    5   6   7   8   9    10   11  12  13  14  15
    ;Inst   start   hold   vel  Ton Oct	Voi Ste	En1  Gls  Ups Ren 2gl 3gl Vol
    1       0	    120     74  201   2   1   8   2    0  -1  2   0   0    10
```
The varliable current_chord contains an array of pitch, velocity for each note in the chord currently being held. 
```text
current_chord = array([[ 53.,  89.],
       [ 57.,  96.],
       [ 59.,  93.],
       [ 60., 103.]])
```

In [11]:
def play_note(thread, pitch_velocity, voice=1, stereo=8, envelope=2, gliss=0, ups=0, renv=2, second_gliss=0, third_gliss=0, volume=10):
    for keys_pressed in pitch_velocity: # one for each note currently being played, with its velocity
        pitch, velocity = keys_pressed
        cent_value = (pitch % 12) * 100
        octave_value = (pitch // 12) 
        velocity = np.clip(velocity,60, 70)
        # play the notes one at a time.
        logging.info(f'playing one note at a time. {pitch = }, {velocity = }, {cent_value = }, {octave_value = }')
        thread.scoreEvent(False, 'i', (1, 0,  10, velocity, cent_value, octave_value, voice, stereo, envelope, gliss, ups, renv, second_gliss, third_gliss, volume))

In [ ]:
# if you have an instance running, or killed the following cell, then you need to close the port and threads out, and delete the csound instance.
port.close()
thread.stop()
thread.join()
del cs

		   overall amps:   3638.6   4422.2
	   overall samples out of range:        0        0
0 errors in performance
4919 512 sample blks of 64-bit floats written to dac


In [16]:
cs, thread = initiate_csound()
csound_duration = 300 # seconds, 5 minutes
port = mido.open_input('LPK25 MIDI 1')
one_hot_note = np.zeros([127]) # one hot encoding of the current chord, with velocity as value
ticks_per_beat = 24 # pulses per quarter note
start = time.time()
print(f'{start = }')
for msg in port:
    logging.info(f'{msg.type = }, {msg.note = }, {msg.velocity = }, time since start: {round((time.time() - start) * ticks_per_beat,1)}') # you can refer to values as msg.something
    if msg.type == 'note_on' and msg.velocity > 0:
        one_hot_note[msg.note] = msg.velocity
        # Only play the note that was just struck
        just_played = np.array([[msg.note, msg.velocity]])
        result = play_note(thread, just_played, voice=5, volume=5)
    elif msg.type == 'note_off' or (msg.type == 'note_on' and msg.velocity == 0):
        one_hot_note[msg.note] = 0
    if time.time() > start + csound_duration: 
        print(f'no longer waiting for a MIDI message, {csound_duration} seconds have passed, so exiting loop.')
        break
port.close()
thread.stop()
thread.join()
del cs

--Csound version 7.0 (double samples) Feb  3 2026
[commit: 4596d8246d1a9df12004e3abfd8d43d2e89284e7]
using libsndfile-1.2.2
sr = 44100.0, kr = 11025.000, ksmps = 4
0dBFS level = 32768.0, A4 tuning = 440.0
audio buffered in 256 sample-frame blocks
writing 512 sample blks of 64-bit floats to dac
SECTION 1:


start = 1773508288.212071


end of Performance
		   overall amps:   4766.9   5789.1
	   overall samples out of range:        0        0
0 errors in performance
51680 512 sample blks of 64-bit floats written to dac


no longer waiting for a MIDI message, 300 seconds have passed, so exiting loop.


In [15]:
port.close()
thread.stop()
thread.join()
del cs

		   overall amps:   3638.6   4422.2
	   overall samples out of range:        0        0
0 errors in performance
4919 512 sample blks of 64-bit floats written to dac


In [ ]:
# what are typical velocity and volume levels used in the perc_part_finger_pianos.npy dataset?
fp_arr = np.load('perc_part_finger_pianos.npy')
vel_pos = 3
vol_pos = 14
print(f'{fp_arr.shape = }, {fp_arr[0] = }')
print(f'velocity values: {np.unique(fp_arr[:,vel_pos], return_counts=True)}')
print(f'volume values: {np.unique(fp_arr[:,vol_pos], return_counts=True)}')

fp_arr.shape = (4313, 15), fp_arr[0] = array([  1.   ,   5.25 ,   5.303,  74.   , 201.   ,   2.   ,   0.   ,
         8.   ,   2.   ,   0.   ,  -1.   ,   2.   ,   0.   ,   0.   ,
        10.   ])
velocity values: (array([69., 71., 72., 74., 75., 77.]), array([ 575,  575,  959, 1065,  512,  627]))
volume values: (array([ 0.,  0.,  0.,  0.,  1.,  1.,  1.,  2.,  3.,  3.,  4.,  4.,  5.,
       10., 11.]), array([891, 270, 356, 325,  90, 127, 117, 367, 341, 119, 111, 304,  80,
       617, 198]))


In [68]:
voice_time = atu.init_voice_time()
for instrument_name in voice_time.keys():
    print(f'{instrument_name = }, {voice_time[instrument_name]['csound_voice'] = }')

instrument_name = 'fing1', voice_time[instrument_name]['csound_voice'] = 1
instrument_name = 'fing2', voice_time[instrument_name]['csound_voice'] = 1
instrument_name = 'fing3', voice_time[instrument_name]['csound_voice'] = 1
instrument_name = 'bfin1', voice_time[instrument_name]['csound_voice'] = 24
instrument_name = 'fing4', voice_time[instrument_name]['csound_voice'] = 1
instrument_name = 'fing5', voice_time[instrument_name]['csound_voice'] = 1
instrument_name = 'fing6', voice_time[instrument_name]['csound_voice'] = 1
instrument_name = 'fing7', voice_time[instrument_name]['csound_voice'] = 1
instrument_name = 'fing8', voice_time[instrument_name]['csound_voice'] = 1
instrument_name = 'fing9', voice_time[instrument_name]['csound_voice'] = 1
instrument_name = 'fing10', voice_time[instrument_name]['csound_voice'] = 1
instrument_name = 'fing11', voice_time[instrument_name]['csound_voice'] = 1
instrument_name = 'fing12', voice_time[instrument_name]['csound_voice'] = 1
instrument_name = 'fi